In [1]:
%pip install -q langchain langchain-community langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q ollama
%pip install -q chromadb
%pip install -q langchain

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -q langchain-openai langchain_chroma

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -q unstructured

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -q rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install -q JPype1

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install -q konlpy

Note: you may need to restart the kernel to use updated packages.


In [15]:
%pip install -qU langchain-ollama
%pip install -U ollama

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# 문서 로드, 분할

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
from konlpy.tag import Okt

In [9]:
# okt = Okt()

# def len_okt(text):
#     tokens = [token for token in okt.morphs(text)]

#     return len(tokens)

# def okt_tokenize(text):
#     return [token for token in okt.morphs(text)]

# 텍스트 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter

input_chunk_size = 1000
input_chunk_overlap = 100
text_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " ", ""],
    chunk_size = input_chunk_size,
    chunk_overlap = input_chunk_overlap,
    # length_function = len_okt
)

# 경로 지정, 문서 로드
direct = "../../Data_Files"
filetype = 'txt'
print(f"디렉토리 경로'{direct}' 아래 '{filetype}'형식의 파일들을 로드합니다.\n")
loader = DirectoryLoader(direct ,glob="*."+filetype, loader_cls=TextLoader)
docs = loader.load()
print("로드 된 파일 개수 :", len(docs))

# 문서 분할
print(f"청크사이즈{input_chunk_size}, 청크오버랩 사이즈{input_chunk_overlap} 인 text_splitter를 실행합니다.")
texts = text_splitter.split_documents(docs)
print("문서 분할 완료")
print("총 청크 개수: ",len(texts))


디렉토리 경로'../../Data_Files' 아래 'txt'형식의 파일들을 로드합니다.

로드 된 파일 개수 : 1
청크사이즈1000, 청크오버랩 사이즈100 인 text_splitter를 실행합니다.
문서 분할 완료
총 청크 개수:  17


# 임베딩 또는 기존 벡터db사용
- 기존 벡터 db사용 ?가능한가 확인해 봐야함

In [ ]:
# 임베딩
from langchain_community.embeddings import HuggingFaceEmbeddings

model_name = 'nlpai-lab/KURE-v1'
model_name1 = 'KURE-v1' # 폴더 디렉토리에 들어갈 변수, /사용 못함
# model_name = 'jhgan/ko-sbert-nli'
model_kwargs = {'device': 'cpu'} # cpu 사용하려면
# model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True}
print(f"hf_embedding_model설정, 모델 이름 : {model_name}\n")
hf_embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

hf_embedding_model설정, 모델 이름 : nlpai-lab/KURE-v1



C:\Users\User\AppData\Local\Temp\ipykernel_27592\2132665271.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embedding_model = HuggingFaceEmbeddings(


In [12]:
# 임베딩 벡터 경로 지정
# 파일 형식(txt/json)_청크-오버랩_임베딩모델
embedding_directory = '../../Data_Files/Vectorstore'
collectionName = filetype+'_'+str(input_chunk_size)+'-'+str(input_chunk_overlap)+'_'+model_name1
print(f"임베딩 벡터db 경로: {embedding_directory}")
print(f"벡터db collectionName: {collectionName}")


임베딩 벡터db 경로: ../../Data_Files/Vectorstore
벡터db collectionName: txt_1000-100_KURE-v1


In [13]:
import os
from langchain_chroma import Chroma
# save_directory 가 존재한다면 해당 경로를 db로 설정
if os.path.exists(embedding_directory):
    # 기존 DB 로드
    db = Chroma(collection_name=collectionName,embedding_function=hf_embedding_model,persist_directory=embedding_directory)
    print("기존 Chroma DB를 로드했습니다.")
else:
    # 새 DB 생성 및 저장
    db = Chroma.from_documents(texts, hf_embedding_model, persist_directory=embedding_directory)
    print("새로운 Chroma DB를 생성하고 저장했습니다.")

기존 Chroma DB를 로드했습니다.


# 검색 retriever

In [14]:
# 문서 검색기 1
k_num = 3
retriever = db.as_retriever(
    search_kwargs = {'k': k_num}
)

# 문서 검색기 2
from langchain_community.retrievers import BM25Retriever
bm_k_num = 1
bm_retriever = BM25Retriever.from_documents(
    documents = docs,
    # preprocess_func = okt_tokenize,
)

bm_retriever.k = bm_k_num

# 앙상블 retriever
from langchain.retrievers import EnsembleRetriever
retriever1_rate = 0.5
ensemble_retriever = EnsembleRetriever(
    retrievers = [retriever, bm_retriever],
    weights = [retriever1_rate, 1-retriever1_rate]
)

# 검색 문서 포맷
def format_docs(docs):
    return "\n\n".join(document.page_content for document in docs)

# 프롬프트, LLM 모델 지정

In [15]:
# 프롬프트
from langchain_core.prompts import ChatPromptTemplate

prompt_content = """
            You are an assistant for question-answering tasks.
            Use the following pieces of retrieved context to answer the question.
            If you don't know the answer, just say that you don't know.

            Answer in Korean.

            #Context:
            {context}
            """

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            prompt_content,
        ), ("human", "{question}"),
    ]
)

In [16]:
# from langchain_openai import ChatOpenAI
# from langchain_community.llms import OllamaLLM
from langchain_ollama import ChatOllama
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

LLM_model="gemma3:4b"
llm = ChatOllama(
    model = LLM_model,
    temperature = 0,
    streaming = True,
    callback_manager = CallbackManager([StreamingStdOutCallbackHandler()],)
)

# LLM 체인

In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

chain = {
    "context" : retriever | RunnableLambda(format_docs),
    "question" : RunnablePassthrough()
} | prompt | llm | StrOutputParser()

In [ ]:
question = 'QA엔지니어를 뽑는 채용공고를 알려줘.'
response = chain.invoke(question)

# 평가